In [ ]:
# Upload the tadawul_merged.csv file via Colab's file picker
import pandas as pd

df = pd.read_csv("tadawul_merged.csv")

In [ ]:
# calculates rev and net income using a groupby function
retail_summary = df[(df["profile_sector"] == "Retail trade")  &
 (df["fiscal_year"] == 2025) ].groupby("profile_sector").agg(avg_revenue=("revenue", "mean"),
                                                           med_revenue=("revenue", "median"),
                                                           agr_revenue=("revenue", 'sum'),
                                                           avg_net_income=("net_income","mean"),
                                                           med_net_income=("net_income", "median"),
                                                           agr_net_income=("net_income", "sum")
)

In [ ]:
# Calculate aggregated revenue grouped by fiscal year and profile sector
sector_revenue = df.groupby(['fiscal_year', 'profile_sector']).agg(
    avg_revenue=('revenue', 'mean'),
    med_revenue=('revenue', 'median'),
    total_revenue=('revenue', 'sum'),
    count=('revenue', 'count')
).reset_index()

display(sector_revenue.head(20))

In [ ]:
# Calculate aggregated revenue grouped by fiscal year and profile industry
industry_revenue = df.groupby(['fiscal_year', 'profile_industry']).agg(
    avg_revenue=('revenue', 'mean'),
    med_revenue=('revenue', 'median'),
    total_revenue=('revenue', 'sum'),
    count=('revenue', 'count')
).reset_index()

display(industry_revenue.head(20))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# =========================================================
# 1. CLEAN THE DATA USING THE COURSE APPROACH
# =========================================================
# Work from the original dataframe so this cell can be rerun safely.
clean_df = df.copy()
clean_df.columns = clean_df.columns.str.strip().str.lower()

# Remove exact duplicate rows first.
clean_df = clean_df.drop_duplicates().copy()

# Remove repeated ticker-year observations.
# The course identified duplicate ticker-year rows in the raw file.
# When period_end_date is available, keep the latest observation for that firm-year.
if 'period_end_date' in clean_df.columns:
    clean_df['period_end_date'] = pd.to_datetime(clean_df['period_end_date'], errors='coerce')
    clean_df = (
        clean_df
        .sort_values(['ticker', 'fiscal_year', 'period_end_date'])
        .drop_duplicates(subset=['ticker', 'fiscal_year'], keep='last')
        .copy()
    )
else:
    clean_df = (
        clean_df
        .sort_values(['ticker', 'fiscal_year'])
        .drop_duplicates(subset=['ticker', 'fiscal_year'], keep='last')
        .copy()
    )

# Use the actual Tadawul column names and standardize them for the dashboard.
column_map = {
    'revenue': 'revenue',
    'net_income': 'net_income',
    'operating_income': 'operating_income',
    'ebitda': 'ebitda',
    'assets': 'bs_tot_asset',
    'liabilities': 'bs_tot_liab',
    'equity': 'bs_total_equity'
}

for dashboard_col, source_col in column_map.items():
    if source_col in clean_df.columns:
        clean_df[dashboard_col] = pd.to_numeric(clean_df[source_col], errors='coerce')
    else:
        clean_df[dashboard_col] = np.nan

# Convert grouping fields and keep only valid firm-year observations.
clean_df['fiscal_year'] = pd.to_numeric(clean_df['fiscal_year'], errors='coerce')
clean_df = clean_df.dropna(subset=['ticker', 'fiscal_year']).copy()
clean_df['fiscal_year'] = clean_df['fiscal_year'].astype(int)

# Replace +/- infinity with NaN. Never silently convert missing financial data to zero.
financial_cols = list(column_map.keys())
clean_df[financial_cols] = clean_df[financial_cols].replace([np.inf, -np.inf], np.nan)

# Keep companies/firm-years in the panel even when a specific financial item is missing.
# Eligibility is handled later based on the user's selected scoring features.

# Revenue <= 0 is not economically meaningful for margin/turnover or YoY growth in this screen.
clean_df.loc[clean_df['revenue'] <= 0, 'revenue'] = np.nan

# =========================================================
# 2. FINANCIAL RATIOS
# =========================================================
def safe_div_positive_den(num, den):
    """Divide only when the denominator is strictly positive.

    This avoids economically misleading ratios such as ROE or debt/equity
    when book equity is zero/negative, and avoids margin/turnover on
    non-positive revenue/assets. Missing/invalid values remain NaN.
    """
    valid_den = den.where(den > 0, np.nan)
    result = num / valid_den
    return result.replace([np.inf, -np.inf], np.nan)

clean_df['return_on_assets'] = safe_div_positive_den(clean_df['net_income'], clean_df['assets'])
clean_df['return_on_equity'] = safe_div_positive_den(clean_df['net_income'], clean_df['equity'])
clean_df['profit_margin'] = safe_div_positive_den(clean_df['net_income'], clean_df['revenue'])
clean_df['asset_turnover'] = safe_div_positive_den(clean_df['revenue'], clean_df['assets'])
clean_df['debt_to_equity'] = safe_div_positive_den(clean_df['liabilities'], clean_df['equity'])
clean_df.loc[clean_df['liabilities'] < 0, 'debt_to_equity'] = np.nan

ratio_cols = [
    'return_on_assets',
    'return_on_equity',
    'profit_margin',
    'asset_turnover',
    'debt_to_equity'
]

# =========================================================
# 3. YOY CHANGES — GROUP BY TICKER FIRST
# =========================================================
metric_cols = [
    'revenue',
    'operating_income',
    'ebitda',
    'net_income',
    'assets',
    'liabilities',
    'equity'
]

clean_df = clean_df.sort_values(['ticker', 'fiscal_year']).copy()

# Compute true year-over-year changes only:
#   1) within the same ticker,
#   2) when the previous observation is exactly the prior fiscal year, and
#   3) when the prior-year base is positive.
# This prevents sign-crossing/zero-base percentages from creating meaningless extremes.
firm_group = clean_df.groupby('ticker', group_keys=False)
previous_year = firm_group['fiscal_year'].shift(1)
consecutive_year = clean_df['fiscal_year'].eq(previous_year + 1)

for col in metric_cols:
    yoy_col = f'{col}_yoy'
    previous_value = firm_group[col].shift(1)
    current_value = clean_df[col]

    valid_growth = (
        consecutive_year
        & previous_value.notna()
        & current_value.notna()
        & (previous_value > 0)
    )

    growth = (current_value / previous_value) - 1
    clean_df[yoy_col] = (
        growth.where(valid_growth)
        .replace([np.inf, -np.inf], np.nan)
    )

# =========================================================
# 4. WINSORIZE 1st/99th PERCENTILE WITHIN EACH YEAR,
#    THEN Z-SCORE NORMALIZE WITHIN EACH YEAR
# =========================================================
def winsorize_1_99(series):
    s = series.replace([np.inf, -np.inf], np.nan)
    valid = s.dropna()
    if len(valid) < 4:
        return s
    lower = valid.quantile(0.01)
    upper = valid.quantile(0.99)
    return s.clip(lower=lower, upper=upper)


def zscore(series):
    s = series.replace([np.inf, -np.inf], np.nan)
    std = s.std()
    if pd.isna(std) or std == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / std

z_df = clean_df.copy()
all_raw_features = ratio_cols + [f'{m}_yoy' for m in metric_cols]

for col in all_raw_features:
    winsor_col = f'{col}_winsor'
    z_col = f'{col}_z'

    z_df[winsor_col] = (
        z_df.groupby('fiscal_year')[col]
        .transform(winsorize_1_99)
    )

    z_df[z_col] = (
        z_df.groupby('fiscal_year')[winsor_col]
        .transform(zscore)
        .replace([np.inf, -np.inf], np.nan)
    )

# Direction adjustment: every score column is defined so HIGHER = BETTER.
# Lower leverage is better, and slower liabilities growth is treated as financially stronger.
feature_direction = {
    'debt_to_equity_z': -1,
    'liabilities_yoy_z': -1
}

for col in [f'{r}_z' for r in ratio_cols] + [f'{m}_yoy_z' for m in metric_cols]:
    score_col = col.replace('_z', '_score')
    direction = feature_direction.get(col, 1)
    z_df[score_col] = z_df[col] * direction

# =========================================================
# 5. USER-FRIENDLY DASHBOARD CONTROLS — SAME WORKFLOW
# =========================================================
metric_labels = {
    'revenue': 'Revenue growth',
    'operating_income': 'Operating income growth',
    'ebitda': 'EBITDA growth',
    'net_income': 'Net income growth',
    'assets': 'Asset growth',
    'liabilities': 'Liabilities growth (lower is better)',
    'equity': 'Equity growth'
}

ratio_labels = {
    'return_on_assets': 'Return on assets (ROA)',
    'return_on_equity': 'Return on equity (ROE)',
    'profit_margin': 'Net profit margin',
    'asset_turnover': 'Asset turnover',
    'debt_to_equity': 'Debt-to-equity (lower is better)'
}

available_metrics = list(metric_labels.keys())
available_ratios = list(ratio_labels.keys())

sectors = ['All'] + sorted(z_df['profile_sector'].dropna().astype(str).unique().tolist())
industries = ['All'] + sorted(z_df['profile_industry'].dropna().astype(str).unique().tolist())
years = sorted(z_df['fiscal_year'].dropna().unique().tolist())

def metric_display_name(x):
    return metric_labels.get(x, x)

def ratio_display_name(x):
    return ratio_labels.get(x, x)

metrics_sel = widgets.SelectMultiple(
    options=[(metric_display_name(x), x) for x in available_metrics],
    value=('revenue', 'net_income'),
    description='FS Metrics',
    rows=7,
    layout=widgets.Layout(width='360px')
)

ratios_sel = widgets.SelectMultiple(
    options=[(ratio_display_name(x), x) for x in available_ratios],
    value=('return_on_assets', 'profit_margin'),
    description='Fin Ratios',
    rows=7,
    layout=widgets.Layout(width='360px')
)

year_slider = widgets.SelectMultiple(
    options=years,
    value=(2025,) if 2025 in years else (max(years),),
    description='Years',
    rows=7,
    layout=widgets.Layout(width='230px')
)

sector_filter = widgets.Dropdown(
    options=sectors,
    value='All',
    description='Sector',
    layout=widgets.Layout(width='420px')
)

industry_filter = widgets.Dropdown(
    options=industries,
    value='All',
    description='Industry',
    layout=widgets.Layout(width='420px')
)

weights_container = widgets.VBox([])
weight_sum_label = widgets.HTML()
btn_recalc = widgets.Button(
    description='Calculate & Render',
    button_style='primary',
    icon='bar-chart',
    layout=widgets.Layout(width='220px', height='42px')
)
output_area = widgets.Output()


def get_selected_feature_specs():
    specs = []
    for m in metrics_sel.value:
        specs.append({
            'raw_name': f'{m}_yoy',
            'score_name': f'{m}_yoy_score',
            'label': metric_labels[m]
        })
    for r in ratios_sel.value:
        specs.append({
            'raw_name': r,
            'score_name': f'{r}_score',
            'label': ratio_labels[r]
        })
    return specs


def get_current_weight_sum():
    total = 0.0
    for box in weights_container.children:
        if len(box.children) >= 2:
            control_row = box.children[1]
            if len(control_row.children) >= 2:
                total += float(control_row.children[1].value)
    return total


def get_weight_dict():
    weights = {}
    for box in weights_container.children:
        if len(box.children) >= 2:
            text_widget = box.children[1].children[1]
            feature_name = getattr(text_widget, 'feature_name', None)
            if feature_name:
                weights[feature_name] = float(text_widget.value)
    return weights


def check_weights(*args):
    total = get_current_weight_sum()
    if np.isclose(total, 1.0):
        weight_sum_label.value = (
            f"<b style='color:#188038'>✓ Weight total: {total:.2f}</b>"
        )
        btn_recalc.disabled = False
    else:
        weight_sum_label.value = (
            f"<b style='color:#d93025'>Weight total: {total:.2f} — must equal 1.00</b>"
        )
        btn_recalc.disabled = True


def update_weights_ui(*args):
    specs = get_selected_feature_specs()
    children = []

    default_weight = 1.0 / len(specs) if specs else 0.0

    for spec in specs:
        label = widgets.HTML(
            value=f"<b>{spec['label']}</b>",
            layout=widgets.Layout(width='330px')
        )
        slider = widgets.FloatSlider(
            value=default_weight,
            min=0.0,
            max=1.0,
            step=0.01,
            readout=False,
            continuous_update=False,
            layout=widgets.Layout(width='390px')
        )
        text_input = widgets.FloatText(
            value=default_weight,
            step=0.01,
            layout=widgets.Layout(width='90px')
        )
        text_input.feature_name = spec['score_name']

        widgets.jslink((slider, 'value'), (text_input, 'value'))
        slider.observe(check_weights, names='value')
        text_input.observe(check_weights, names='value')

        children.append(
            widgets.VBox([
                label,
                widgets.HBox([slider, text_input])
            ], layout=widgets.Layout(margin='4px 0 8px 0'))
        )

    weights_container.children = tuple(children)
    check_weights()


metrics_sel.observe(update_weights_ui, names='value')
ratios_sel.observe(update_weights_ui, names='value')
update_weights_ui()

# =========================================================
# 6. RECALCULATE SCORE AND RANK WHEN USER CLICKS THE BUTTON
# =========================================================
def process_and_draw(b):
    with output_area:
        clear_output(wait=True)

        selected_years = sorted(list(year_slider.value))
        if not selected_years:
            print('Please select at least one fiscal year.')
            return

        weight_dict = get_weight_dict()
        if not weight_dict:
            print('Please select at least one metric or ratio.')
            return

        if not np.isclose(sum(weight_dict.values()), 1.0):
            print('Weights must sum to exactly 1.00.')
            return

        # Start with selected years. Keep global market scores/ranks before dashboard filters.
        score_df = z_df[z_df['fiscal_year'].isin(selected_years)].copy()

        required_features = list(weight_dict.keys())
        score_df = score_df.replace([np.inf, -np.inf], np.nan)
        score_df = score_df.dropna(subset=required_features).copy()

        if score_df.empty:
            print('No complete observations are available for the selected features and years.')
            return

        # Composite score: selected normalized feature scores x user-defined weights.
        score_df['final_score'] = 0.0
        for col, weight in weight_dict.items():
            score_df['final_score'] += score_df[col] * weight

        # Market-wide rank is calculated BEFORE sector / industry display filters.
        score_df['market_rank'] = (
            score_df.groupby('fiscal_year')['final_score']
            .rank(method='min', ascending=False)
        )

        score_df['sector_rank'] = (
            score_df.groupby(['fiscal_year', 'profile_sector'])['final_score']
            .rank(method='min', ascending=False)
        )

        score_df['industry_rank'] = (
            score_df.groupby(['fiscal_year', 'profile_industry'])['final_score']
            .rank(method='min', ascending=False)
        )

        # Apply the user's view filters after ranking.
        view_df = score_df.copy()
        if sector_filter.value != 'All':
            view_df = view_df[view_df['profile_sector'].astype(str) == str(sector_filter.value)]
        if industry_filter.value != 'All':
            view_df = view_df[view_df['profile_industry'].astype(str) == str(industry_filter.value)]

        if view_df.empty:
            print('No data are available for the selected sector / industry filters.')
            return

        latest_year = max(selected_years)
        latest_set = view_df[view_df['fiscal_year'] == latest_year].copy()

        if latest_set.empty:
            print(f'No eligible firms found for {latest_year}.')
            return

        latest_set = latest_set.sort_values('final_score', ascending=False)
        latest_set['view_rank'] = np.arange(1, len(latest_set) + 1)
        top_latest = latest_set.head(10).copy()
        top_tickers = top_latest['ticker'].tolist()

        # Coverage KPI
        total_firms = z_df[z_df['fiscal_year'] == latest_year]['ticker'].nunique()
        eligible_market_firms = score_df[score_df['fiscal_year'] == latest_year]['ticker'].nunique()
        visible_firms = latest_set['ticker'].nunique()

        display(HTML(
            f"""
            <div style='display:flex; gap:18px; margin:8px 0 14px 0;'>
              <div style='padding:12px 18px;border:1px solid #ddd;border-radius:8px;'><b>Fiscal Year</b><br><span style='font-size:24px'>{latest_year}</span></div>
              <div style='padding:12px 18px;border:1px solid #ddd;border-radius:8px;'><b>Eligible Market Firms</b><br><span style='font-size:24px'>{eligible_market_firms}/{total_firms}</span></div>
              <div style='padding:12px 18px;border:1px solid #ddd;border-radius:8px;'><b>Firms in Current View</b><br><span style='font-size:24px'>{visible_firms}</span></div>
            </div>
            """
        ))

        # Ranked table
        company_col = 'profile_name' if 'profile_name' in top_latest.columns else 'ticker'
        display_cols = [
            'view_rank', 'market_rank', 'ticker', company_col,
            'profile_sector', 'profile_industry', 'final_score'
        ]

        ranked_table = top_latest[display_cols].copy()
        ranked_table = ranked_table.rename(columns={
            'view_rank': 'View Rank',
            'market_rank': 'Market Rank',
            'ticker': 'Ticker',
            company_col: 'Company',
            'profile_sector': 'Sector',
            'profile_industry': 'Industry',
            'final_score': 'Score'
        })

        print(f'--- Top 10 Firms — {latest_year} ---')
        display(
            ranked_table.style
            .format({'Score': '{:.3f}', 'Market Rank': '{:.0f}', 'View Rank': '{:.0f}'})
        )

        # Show exactly what the user weighted.
        weight_rows = []
        score_to_label = {spec['score_name']: spec['label'] for spec in get_selected_feature_specs()}
        for feature, weight in weight_dict.items():
            weight_rows.append({
                'Selected Factor': score_to_label.get(feature, feature),
                'Weight': weight
            })
        weight_table = pd.DataFrame(weight_rows)
        print('Selected scoring model:')
        display(weight_table.style.format({'Weight': '{:.0%}'}))

        # Chart: one year = ranked bar chart; multiple years = score trend for current top 10.
        plt.figure(figsize=(13, 6))

        if len(selected_years) == 1:
            chart_labels = (
                top_latest['profile_name'].fillna(top_latest['ticker'].astype(str))
                if 'profile_name' in top_latest.columns
                else top_latest['ticker'].astype(str)
            )
            plt.bar(chart_labels, top_latest['final_score'])
            plt.title(f'Top 10 Composite Scores — {latest_year}')
            plt.ylabel('Composite Weighted Score')
            plt.xlabel('Company')
            plt.xticks(rotation=45, ha='right')
        else:
            trend_df = view_df[
                (view_df['ticker'].isin(top_tickers)) &
                (view_df['fiscal_year'].isin(selected_years))
            ].copy()

            name_map = {}
            if 'profile_name' in trend_df.columns:
                name_map = (
                    trend_df.dropna(subset=['profile_name'])
                    .drop_duplicates('ticker')
                    .set_index('ticker')['profile_name']
                    .to_dict()
                )

            for ticker in top_tickers:
                one = trend_df[trend_df['ticker'] == ticker].sort_values('fiscal_year')
                if not one.empty:
                    label = name_map.get(ticker, str(ticker))
                    plt.plot(one['fiscal_year'], one['final_score'], marker='o', label=label)

            plt.title('Composite Score Trend — Current Top 10')
            plt.xlabel('Fiscal Year')
            plt.ylabel('Composite Weighted Score')
            plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.grid(True, alpha=0.25)

        plt.tight_layout()
        plt.show()


btn_recalc.on_click(process_and_draw)

# =========================================================
# 7. RENDER THE DASHBOARD
# =========================================================
dashboard_header = widgets.HTML(
    """
    <div style='padding:14px 4px 8px 4px;'>
      <h2 style='margin:0'>Firm Ranking Dashboard</h2>
      <p style='margin:5px 0 0 0;color:#555'>
        Select financial metrics and ratios, assign weights that sum to 1.00, then recalculate.
        Data are deduplicated, cleaned of NaN/inf for selected features, winsorized at the 1st/99th percentile within each year, and normalized by fiscal year.
      </p>
    </div>
    """
)

controls = widgets.VBox([
    widgets.HBox([metrics_sel, ratios_sel, year_slider]),
    widgets.HBox([sector_filter, industry_filter]),
    widgets.HTML('<h4 style="margin-bottom:4px">Assign Weights</h4>'),
    weights_container,
    weight_sum_label,
    btn_recalc
])

display(widgets.VBox([
    dashboard_header,
    controls,
    output_area
]))
